# Training_File_Architecture.ipynb — STEP 2E Architecture Exploration

**Branch git**: `Architecture_Exploration` (creato da `main` post-merge STEP 2D).

**Contesto post-P14**: il floor decomposition ha mostrato che **78% del val~0.28 è "residuo architettura"** (F7=0.2198). Il limite NON è ottimizzatore, capacità, dataset size o vincoli FPGA. È la topologia.

Questo notebook esplora **8 varianti architetturali** preservando tutti i vincoli FPGA: `po2_quantize`, `SurrogateSpike_Hardware`, bit-shift leak, pesi [2⁻⁴, 2¹].

**Benchmark**: F7 baseline (no OU, no SR, no Po2) → val=0.2198. Ogni variante usa stesso setup ma con `po2_enabled=1` (deploy-realistic).

**Reference val target**: F7=0.2198 (cache `cache_1500_highway_cut0.0_ou0.0.pt` riusata).

| # | Variant | params est. | Razionale |
|---|---------|-------------|-----------|
| A1 | baseline           | 864   | Control, == F7 setup |
| A2 | stacked_2          | 2464  | 2 hidden ALIF (depth) |
| A3 | stacked_2_skip     | 2624  | A2 + MS-style membrane skip |
| A4 | stacked_3_thin     | 2376  | 3 hidden ALIF (depth vs width) |
| A5 | max_delay_12       | 864   | Memoria temporale più lunga (delays) |
| A6 | multi_rate (INNO1) | 864   | bit_shifts=[2,3,4] gerarchia temporale |
| A7 | wta                | 896   | Lateral inhibition (1 neurone inh) |
| A8 | attn (INNO2)       | 3936  | Spike attention lite (Q/K/V Po2, 2 heads) |

**Esiti possibili**:
- best < 0.18 → **breakthrough** architettura, candidato deploy
- 0.18 ≤ best < 0.21 → migliore + considerare combo top-2
- best ≥ 0.22 → architettura attuale near-optimal → procedere F2 (EventProp)

**Tempo stimato Azure CPU**: ~50 min × 8 = ~7h totali.

In [ ]:
# ===========================================================
# CELLA 0 — Bootstrap deps + git checkout Architecture_Exploration
# ===========================================================
import sys

print('Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas
!{sys.executable} -m pip install --quiet nbstripout

# nbstripout per evitare conflitti git sui notebook (output stripping)
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   [OK] deps installed + nbstripout attivato')

# Sync con remote
print('\ngit fetch + checkout Architecture_Exploration + pull:')
!git fetch origin
!git checkout Architecture_Exploration 2>/dev/null || git checkout -b Architecture_Exploration origin/Architecture_Exploration
!git pull --no-rebase --no-edit origin Architecture_Exploration

print('\nBranch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 — ENV CHECK + verifica --arch_variant CLI
# ===========================================================
import sys, platform, os

print(f'Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   [OK] {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   [FAIL] {mod:<14} {e}')
        checks.append((mod, False))

import torch
if torch.cuda.is_available():
    print(f'\nCUDA:            [OK] {torch.cuda.get_device_name(0)}')
else:
    print(f'\nCUDA:            [INFO] training su CPU')

print('\nFile critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'core/neurons.py', 'data/generator.py']:
    exists = os.path.isfile(f)
    print(f'   {"[OK]" if exists else "[MISSING]"} {f}')
    checks.append((f, exists))

# Verifica nuovo CLI --arch_variant
import subprocess
res = subprocess.run([sys.executable, 'train.py', '--help'], capture_output=True, text=True)
has_av = '--arch_variant' in (res.stdout + res.stderr)
print(f'\ntrain.py supporta --arch_variant: {"[OK]" if has_av else "[MISSING] (branch sbagliato? rifare Cella 0)"}')
checks.append(('arch_variant CLI', has_av))

# Verifica build_model importabile + tutte le 8 varianti costruiscono
try:
    from core.network import build_model
    VARIANTS_CHECK = ['baseline','stacked_2','stacked_2_skip','stacked_3_thin',
                      'max_delay_12','multi_rate','wta','attn']
    for v in VARIANTS_CHECK:
        m = build_model(v)
        n = sum(p.numel() for p in m.parameters())
        print(f'   [OK] variant {v:18s} params={n:>6,d}')
    checks.append(('build_model 8 variants', True))
except Exception as e:
    print(f'   [FAIL] build_model: {e}')
    checks.append(('build_model 8 variants', False))

n_fail = sum(1 for _, ok in checks if not ok)
if n_fail == 0:
    print('\n[OK] ENV CHECK PASSED -- procedi con Cella 2')
else:
    print(f'\n[FAIL] ENV CHECK FAILED -- {n_fail} problemi')
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 — ARCH_PLAN: 8 varianti su cache F2 (deploy-realistic con Po2 ON)
# ===========================================================
# Tutte le varianti condividono lo stesso setup di F7 (no OU, no SR) ma con
# po2_enabled=1 (= deploy FPGA real). Cache F2 ('cache_1500_highway_cut0.0_ou0.0.pt')
# è gia' disponibile da Floor_Diagnostic — riusata da tutti gli 8 run.

import time, subprocess, sys, os, datetime, shutil, glob, json
import pandas as pd, numpy as np

ARCH_COMMON = {
    'n_train':              1500,
    'n_val':                300,
    'seq_len':              50,
    'scenario_mix':         'highway',
    'cut_in_ratio':         0.0,
    # capacity default (override per stacked_3_thin che usa 24/6)
    'cf_hidden_size':       32,
    'cf_rank':              8,
    # PINN weights default (lambda_sr=0 per pulizia, come F7)
    'lambda_data':          1.0,
    'lambda_phys':          0.1,
    'lambda_ou':            0.05,
    'lambda_bc':            1.0,
    'lambda_sr':            0.0,        # come F7
    'noise_scale':          0.0,        # come F2/F7 (cache esistente)
    'po2_enabled':          1,          # deploy-realistic
    # AdamW vincente STEP 2C
    'optimizer':            'adamw',
    'lr':                   2e-3,
    'batch_size':           8,
    'val_batch_size':       64,
    'scheduler':            'onecycle',
    'max_lr':               2e-3,
    'epochs':               15,
    'max_steps_per_epoch':  190,        # = STEP 2C Plan B = F7
    # Safety: no abort, no early stop
    'max_inf_streak':       99999,
    'early_stop_patience':  0,
    'early_stop_delta':     0.005,
}

ARCH_PLAN = [
    {'tag': 'P15_S2E_A1_baseline',       'arch_variant': 'baseline'},
    {'tag': 'P15_S2E_A2_stacked_2',      'arch_variant': 'stacked_2'},
    {'tag': 'P15_S2E_A3_stacked_2_skip', 'arch_variant': 'stacked_2_skip'},
    {'tag': 'P15_S2E_A4_stacked_3_thin', 'arch_variant': 'stacked_3_thin'},
    {'tag': 'P15_S2E_A5_max_delay_12',   'arch_variant': 'max_delay_12'},
    {'tag': 'P15_S2E_A6_multi_rate',     'arch_variant': 'multi_rate'},
    {'tag': 'P15_S2E_A7_wta',            'arch_variant': 'wta'},
    {'tag': 'P15_S2E_A8_attn',           'arch_variant': 'attn'},
]

# Quali eseguire? Modifica per disabilitare singoli plan.
RUN_ARCH = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8']

def _cache_path(plan):
    full = {**ARCH_COMMON, **plan}
    base = f'data/cache_{full["n_train"]}_{full["scenario_mix"]}_cut{full["cut_in_ratio"]}'
    if full['noise_scale'] != 1.0:
        base += f'_ou{full["noise_scale"]}'
    return f'{base}.pt'

print(f'ARCH EXPLORATION: {len(ARCH_PLAN)} varianti')
print(f'Reference F7 (no-OU no-SR no-Po2): val=0.2198 (cache riusata)')
print(f'Tutte le varianti hanno po2_enabled=1 (deploy-realistic).')
print(f'Safety: max_inf_streak=99999, early_stop=0')

from core.network import build_model
for i, plan in enumerate(ARCH_PLAN, 1):
    label = f'A{i}'
    active = '[RUN]' if label in RUN_ARCH else '[SKIP]'
    full = {**ARCH_COMMON, **plan}
    cp = _cache_path(plan)
    cache_exists = 'esistente' if os.path.isfile(cp) else 'da generare'
    n_steps = full['max_steps_per_epoch'] * full['epochs']
    # Stima params via build_model (no kwargs override per stacked_3_thin)
    try:
        m_tmp = build_model(plan['arch_variant'])
        n_params = sum(p.numel() for p in m_tmp.parameters())
        del m_tmp
    except Exception as e:
        n_params = -1
    print(f'\n  {active} {label}: {full["tag"]}  variant={plan["arch_variant"]}')
    print(f'     params: {n_params:,}  budget: {n_steps} step ({full["max_steps_per_epoch"]}/ep x {full["epochs"]} ep)')
    print(f'     cache: {cp}  ({cache_exists})')

In [ ]:
# ===========================================================
# CELLA 3 — RUN sweep architetture (continue-on-failure)
# ===========================================================
# SKIP_IF_EXISTS=True (default): salta plan i cui results/<tag>/ esistono gia'.
# Permette di rilanciare la cella dopo crash/interruzioni senza ripetere lavoro.
SKIP_IF_EXISTS = True

def _build_cli_arch(plan):
    full = {**ARCH_COMMON, **plan}
    args = [
        sys.executable, 'train.py',
        '--arch_variant',        plan['arch_variant'],
        '--epochs',              str(full['epochs']),
        '--n_train',             str(full['n_train']),
        '--n_val',               str(full['n_val']),
        '--batch_size',          str(full['batch_size']),
        '--val_batch_size',      str(full['val_batch_size']),
        '--seq_len',             str(full['seq_len']),
        '--scheduler',           full['scheduler'],
        '--max_lr',              str(full['max_lr']),
        '--lr',                  str(full['lr']),
        '--optimizer',           full['optimizer'],
        '--scenario_mix',        full['scenario_mix'],
        '--cut_in_ratio',        str(full['cut_in_ratio']),
        '--cf_hidden_size',      str(full['cf_hidden_size']),
        '--cf_rank',             str(full['cf_rank']),
        '--lambda_data',         str(full['lambda_data']),
        '--lambda_phys',         str(full['lambda_phys']),
        '--lambda_ou',           str(full['lambda_ou']),
        '--lambda_bc',           str(full['lambda_bc']),
        '--lambda_sr',           str(full['lambda_sr']),
        '--noise_scale',         str(full['noise_scale']),
        '--po2_enabled',         str(full['po2_enabled']),
        '--max_inf_streak',      str(full['max_inf_streak']),
        '--early_stop_patience', str(full['early_stop_patience']),
        '--early_stop_delta',    str(full['early_stop_delta']),
        '--max_steps_per_epoch', str(full['max_steps_per_epoch']),
        '--data_cache',          _cache_path(plan),
        '--tag',                 full['tag'],
    ]
    return args

def _push_arch_result(plan):
    full = {**ARCH_COMMON, **plan}
    tag = full['tag']
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN] {src} mancante')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            val_str = f'best val={edf.val_total.min():.4f} (E{int(edf.val_total.idxmin())+1}/{len(edf)})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (S2E Architecture): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'variant={plan["arch_variant"]}, po2_enabled={full["po2_enabled"]}, noise_scale={full["noise_scale"]}\n'
        f'Branch Architecture_Exploration\n'
    )
    with open('/tmp/arch_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/arch_msg.txt
    !rm /tmp/arch_msg.txt
    !git pull --no-rebase --no-edit origin Architecture_Exploration
    !git push origin Architecture_Exploration
    return True

# Esecuzione
arch_results = []
t_start = time.time()

for i, plan in enumerate(ARCH_PLAN, 1):
    label = f'A{i}'
    if label not in RUN_ARCH:
        print(f'\n[SKIP] {label} ({plan["tag"]}) -- non in RUN_ARCH')
        continue
    full = {**ARCH_COMMON, **plan}
    tag  = full['tag']
    if SKIP_IF_EXISTS and os.path.isfile(f'results/{tag}/training_log.csv'):
        try:
            ep_existing = pd.read_csv(f'results/{tag}/training_log.csv')
            v_existing = ep_existing.val_total.min() if len(ep_existing) else None
            v_str = f'val={v_existing:.4f}' if v_existing is not None else 'empty'
        except Exception:
            v_str = 'unreadable'
        print(f'\n[SKIP_EXIST] {label}: results/{tag}/ gia presente ({v_str}). '
              f'Set SKIP_IF_EXISTS=False per ri-eseguire.')
        arch_results.append({'label': label, 'tag': tag, 'status': 'skipped_existing', 'elapsed': 0.0})
        continue
    print('\n' + '=' * 78)
    print(f'{label}: {tag}  variant={plan["arch_variant"]}')
    print('=' * 78)
    cmd = _build_cli_arch(plan)
    print(f'CLI: {" ".join(cmd[2:])}\n')
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail (exit {res.returncode})'
    print(f'\n{label} terminato in {elapsed/60:.1f}min -- exit: {status}')
    print(f'\nPush results/{tag}...')
    _push_arch_result(plan)
    arch_results.append({'label': label, 'tag': tag, 'status': status, 'elapsed': elapsed})

print(f'\n{"="*78}\nARCH SWEEP COMPLETATO in {(time.time()-t_start)/60:.1f} min\n{"="*78}')
for r in arch_results:
    print(f"   {r['label']:<4} {r['tag']:<34} {r['status']:<20} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 4 — ANALISI COMPARATIVA: tabella + Pareto + verdetto
# ===========================================================
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REF_F7_VAL = 0.2198   # benchmark da Floor_Diagnostic F7 (no OU, no SR, no Po2)
TARGET_VAL = 0.18     # target deploy stretch

rows = []
loaded = []
from core.network import build_model

for plan in ARCH_PLAN:
    full = {**ARCH_COMMON, **plan}
    tag = full['tag']
    base = f'results/{tag}'
    if not os.path.isdir(base):
        rows.append({'variant': plan['arch_variant'], 'tag': tag, 'best_val': '-', 'params': '-',
                     'sr_mean': '-', 'd_vs_F7': '-', 'verdict': 'no_data'})
        continue
    ep_csv = f'{base}/training_log.csv'
    if not os.path.isfile(ep_csv):
        continue
    ep = pd.read_csv(ep_csv)
    if len(ep) == 0:
        continue
    best_val = ep.val_total.min()
    sr_mean = ep.spike_rate.mean() if 'spike_rate' in ep.columns else float('nan')
    # n_params
    try:
        m_tmp = build_model(plan['arch_variant'])
        n_params = sum(p.numel() for p in m_tmp.parameters())
        del m_tmp
    except Exception:
        n_params = -1
    delta = best_val - REF_F7_VAL
    if best_val < TARGET_VAL:
        verdict = '*** BREAKTHROUGH ***'
    elif delta < -0.005:
        verdict = 'improves vs F7'
    elif abs(delta) < 0.005:
        verdict = 'comparable to F7'
    else:
        verdict = 'worse than F7'
    rows.append({
        'variant':  plan['arch_variant'],
        'tag':      tag.replace('P15_S2E_', ''),
        'params':   f'{n_params:,}',
        'best_val': f'{best_val:.4f}',
        'sr_mean':  f'{sr_mean:.3f}',
        'd_vs_F7':  f'{delta:+.4f}',
        'verdict':  verdict,
    })
    loaded.append({'plan': plan, 'full': full, 'ep': ep, 'best_val': best_val,
                   'n_params': n_params, 'sr_mean': sr_mean})

df = pd.DataFrame(rows)
df = df.sort_values('best_val')
display(Markdown(f'### Architecture Exploration -- benchmark F7 val={REF_F7_VAL:.4f}, target <{TARGET_VAL}'))
display(df)

if not loaded:
    print('Nessun risultato. Esegui Cella 3 prima.')
else:
    out_dir = f'opt_plots/arch_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}'
    os.makedirs(out_dir, exist_ok=True)

    # ── Plot 1: val_total per epoca, overlay ──────────────────────
    fig, ax = plt.subplots(figsize=(12, 6))
    cmap = plt.cm.tab10(np.linspace(0, 1, len(loaded)))
    for i, d in enumerate(loaded):
        ep = d['ep']
        ax.plot(ep.epoch, ep.val_total, 'o-', linewidth=1.8, markersize=6,
                color=cmap[i], label=d['plan']['arch_variant'])
    ax.axhline(REF_F7_VAL, ls='--', color='red', alpha=0.6, label=f'F7 ref ({REF_F7_VAL:.4f})')
    ax.axhline(TARGET_VAL, ls=':', color='green', alpha=0.6, label=f'target ({TARGET_VAL})')
    ax.set_xlabel('epoch'); ax.set_ylabel('val_total')
    ax.set_title('Architecture sweep -- val_total per epoca')
    ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/A_val_per_epoch.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── Plot 2: Pareto val vs params ──────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, d in enumerate(loaded):
        ax.scatter(d['n_params'], d['best_val'], s=120, color=cmap[i],
                   label=d['plan']['arch_variant'], edgecolors='black', linewidths=0.8)
        ax.annotate(d['plan']['arch_variant'], (d['n_params'], d['best_val']),
                    xytext=(7, 4), textcoords='offset points', fontsize=8)
    ax.axhline(REF_F7_VAL, ls='--', color='red', alpha=0.6, label=f'F7 ref')
    ax.axhline(TARGET_VAL, ls=':', color='green', alpha=0.6, label='target')
    ax.set_xlabel('# parametri'); ax.set_ylabel('best val_total')
    ax.set_title('Pareto: capacity vs accuracy')
    ax.set_xscale('log')
    ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/A_pareto.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── Verdetto ────────────────────────────────────────────────────
    display(Markdown('### Verdetto operativo'))
    best = min(loaded, key=lambda d: d['best_val'])
    bv = best['best_val']
    delta = bv - REF_F7_VAL
    print(f'BEST: {best["plan"]["arch_variant"]:18s} val={bv:.4f}  params={best["n_params"]:,}  d_vs_F7={delta:+.4f}')
    print()
    if bv < TARGET_VAL:
        print(f'*** BREAKTHROUGH ARCHITETTURA: val<{TARGET_VAL} ***')
        print(f'   {best["plan"]["arch_variant"]} candidato per deploy.')
        print(f'   Next: re-test con cache REF (con OU) per deploy-realistic val.')
    elif delta < -0.01:
        print(f'MIGLIORAMENTO: {best["plan"]["arch_variant"]} batte F7 di {-delta:.4f}.')
        print(f'   Considera combo top-2 (es. {best["plan"]["arch_variant"]} + scheduler tuning).')
    elif delta < 0:
        print(f'TIE: best vicino a F7 ({delta:+.4f}). Marginale.')
        print(f'   Procedi con re-test su REF per validazione, o passa a F2 (EventProp).')
    else:
        print(f'NESSUN MIGLIORAMENTO: tutte le 8 varianti >= F7.')
        print(f'   -> Architettura attuale near-optimal per BPTT+surrogate.')
        print(f'   -> Procedere con FUTURE_WORK F2 (EventProp).')
    print(f'\nGrafici: {out_dir}/')